# 01 — Pandas Overview & Essential Mechanics
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Data Engineering, and Analytics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
In technical interviews, interviewers use introductory Pandas questions not just to verify syntax, but to test your understanding of **underlying data structures**, **memory layout**, **computational complexity ($O(1)$ vs $O(N)$)**, and **vectorized execution vs Python bytecode overhead**.

### Core Competencies Tested in this Module:
1. **DataFrame & Series Anatomy**: Two-dimensional labeled tables vs 1D arrays; Index hash-map semantics.
2. **Positional vs Label-based Indexing**: `.iloc` vs `.loc` vs `.at` vs `.iat` (slicing boundaries, endpoint inclusivity).
3. **Metadata & Dimensionality**: `len()` vs `.shape` vs `.size` vs `.count()`.
4. **Boolean Indexing**: Element-wise bitwise operators (`&`, `|`, `~`), operator precedence traps, and `.between()`.
5. **Vectorized String Operations & Cleaning**: Safe, idempotent transformations with `.str` accessors, plus handling modern Arrow/String dtypes.
6. **GroupBy & Aggregations**: The Split-Apply-Combine paradigm, `.count()` vs `.size()`, and multi-metric aggregation.
7. **Interview Corner**: The infamous `SettingWithCopyWarning`, memory optimization via downcasting/categoricals, and view vs copy mechanics.

## 1. Environment Setup & Library Version Check
Technical interviews frequently test your awareness of the differences between **Pandas 1.x** and **Pandas 2.x / 3.x** (e.g., Apache Arrow integration, Copy-on-Write semantics by default, and nullable/native string data types).

In [1]:
import os
import numpy as np
import pandas as pd

print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version:  {np.__version__}")

Pandas Version: 3.0.6
NumPy Version:  2.5.3


## 2. Ingestion & The Role of the Index
We load the `movies.csv` dataset, using the movie **Title** as the row index (`index_col="Title"`).

> 💡 **Interview Note — What is an Index in Pandas?**
> - In Pandas, an `Index` is an immutable, hash-table-backed array of labels.
> - **Lookup Complexity**: Accessing a row by its index label (`df.loc['Avatar']`) is **$O(1)$** amortized time complexity, compared to scanning an unindexed column which is **$O(N)$**.
> - **Automatic Alignment**: The Index enables automatic data alignment across Series and DataFrames during arithmetic operations, joins, and unions.
> - **Gotcha**: If an index has **duplicate labels**, Pandas cannot use a simple hash lookup; `.loc` will return a `DataFrame` instead of a `Series`, and lookup degrades toward $O(N)$ with hash bucket scans!

In [2]:
# Robust loader: loads from local file if available, or falls back to public mirror
csv_path = "movies.csv"
if not os.path.exists(csv_path):
    csv_path = "https://raw.githubusercontent.com/paskhaver/pandas-in-action/master/chapter_01_introducing_pandas/movies.csv"

movies = pd.read_csv(csv_path, index_col="Title")
movies

,Rank,Studio,Gross,Year
Title,,,,
Avengers: Endgame,1,Buena Vista,"$2,796.30",2019
Avatar,2,Fox,"$2,789.70",2009
Titanic,3,Paramount,"$2,187.50",1997
Star Wars: The Force Awakens,4,Buena Vista,"$2,068.20",2015
Avengers: Infinity War,5,Buena Vista,"$2,048.40",2018
...,...,...,...,...
Yogi Bear,778,Warner Brothers,$201.60,2010
Garfield: The Movie,779,Fox,$200.80,2004
Cats & Dogs,780,Warner Brothers,$200.70,2001


## 3. Preliminary Inspection: `head()`, `tail()`, and `sample()`
Inspecting the top and bottom rows helps detect header formatting issues, data ordering, and truncation.

> 💡 **Pro-Tip**: In interviews, don't just use `.head()`. Mentioning `.sample()` shows that you check for random records to detect potential bias (e.g. if the top 100 rows are sorted or have missing data in lower rows).

In [3]:
# Inspect the first 4 rows
movies.head(4)

,Rank,Studio,Gross,Year
Title,,,,
Avengers: Endgame,1,Buena Vista,"$2,796.30",2019
Avatar,2,Fox,"$2,789.70",2009
Titanic,3,Paramount,"$2,187.50",1997
Star Wars: The Force Awakens,4,Buena Vista,"$2,068.20",2015


In [4]:
# Inspect the last 6 rows
movies.tail(6)

,Rank,Studio,Gross,Year
Title,,,,
21 Jump Street,777,Sony,$201.60,2012
Yogi Bear,778,Warner Brothers,$201.60,2010
Garfield: The Movie,779,Fox,$200.80,2004
Cats & Dogs,780,Warner Brothers,$200.70,2001
The Hunt for Red October,781,Paramount,$200.50,1990
Valkyrie,782,MGM,$200.30,2008


In [5]:
# Pro-tip: inspect 3 random rows for unbiased sampling
movies.sample(3, random_state=42)

,Rank,Studio,Gross,Year
Title,,,,
Four Weddings and a Funeral,597,Polygram,$245.70,1994
Unbreakable,589,Buena Vista,$248.10,2000
Twister,209,Warner Brothers,$494.50,1996


## 4. Dimensionality & Metadata: `len()`, `shape`, `size`, `dtypes`, `info()`

### ⚠️ Top Interview Question: `len(df)` vs `df.shape` vs `df.size` vs `df.count()`
| Property | Return Type | What It Measures | Includes `NaN`s? | Time Complexity |
| :--- | :--- | :--- | :--- | :--- |
| `len(df)` | `int` | Number of rows | Yes | $O(1)$ (queries `len(df.index)`) |
| `df.shape` | `tuple` | `(n_rows, n_cols)` | Yes | $O(1)$ |
| `df.size` | `int` | Total number of elements ($rows \times cols$) | Yes | $O(1)$ |
| `df.count()` | `Series` | Number of non-null values *per column* | **No** (ignores NaNs) | $O(N \times C)$ (scans columns) |

In [6]:
# Number of rows (O(1))
len(movies)

782

In [7]:
# (rows, columns) tuple
movies.shape

(782, 4)

In [8]:
# Total cells (rows * columns)
movies.size

3128

In [9]:
# Data type of each column
movies.dtypes

Rank      int64
Studio      str
Gross       str
Year      int64
dtype: object

### 💡 Deep Memory Inspection with `info(memory_usage="deep")`
> **Interview Gotcha**: By default, `df.info()` or `df.memory_usage()` reports shallow memory for `object` dtype columns (it only measures the pointer size, 8 bytes per cell, NOT the actual string contents!). Always pass `memory_usage="deep"` to see the real memory footprint in RAM.

In [10]:
# Deep memory inspection: vital for production optimization questions
movies.info(memory_usage="deep")

<class 'pandas.DataFrame'>
Index: 782 entries, Avengers: Endgame to Valkyrie
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Rank    782 non-null    int64
 1   Studio  782 non-null    str  
 2   Gross   782 non-null    str  
 3   Year    782 non-null    int64
dtypes: int64(2), str(2)
memory usage: 150.3 KB


## 5. Row Selection: Positional (`.iloc`) vs Label-Based (`.loc`)

### ⚠️ Critical Interview Comparison: `.loc` vs `.iloc` vs `.at` vs `.iat`
1. **`.iloc` (Integer-location based)**:
   - 0-indexed positional indexing: `0 <= pos < len(df)`.
   - Slicing `0:5` is **exclusive** of the endpoint `5` (returns 5 items: indices 0, 1, 2, 3, 4), exactly like standard Python lists.
2. **`.loc` (Label based)**:
   - Strictly references index labels.
   - Slicing `"A":"C"` is **INCLUSIVE** of both `"A"` and `"C"`.
3. **`.at` & `.iat`**:
   - High-speed scalar accessors. If you only need a single value from a specific cell, `.at[label, col]` is **10x to 50x faster** than `.loc[label, col]` because it skips Series/Index overhead.

In [11]:
# Positional lookup: retrieve the 500th movie (index 499, 0-indexed)
movies.iloc[499]

Rank           500
Studio         Fox
Gross     $288.30 
Year          2018
Name: Maze Runner: The Death Cure, dtype: object

In [12]:
# Label-based lookup: retrieve specific records by their exact Title index
movies.loc[["Forrest Gump", "101 Dalmatians"]]

,Rank,Studio,Gross,Year
Title,,,,
Forrest Gump,119,Paramount,$677.90,1994
101 Dalmatians,425,Buena Vista,$320.70,1996
101 Dalmatians,708,Buena Vista,$215.90,1961


In [13]:
# Fast scalar lookup using .at:
gross_avatar = movies.at["Avatar", "Gross"]
print(f"Avatar Gross (fast scalar lookup via .at): {gross_avatar}")

Avatar Gross (fast scalar lookup via .at): $2,789.70 


## 6. Sorting: By Values and By Index
Sorting is a common requirement in data preprocessing and ranking questions.

> 💡 **Interview Tip — Sorting Options & Best Practices**:
> - **Multi-column sorting with independent directions**: You can pass lists to both `by` and `ascending`: `by=["Studio", "Year"], ascending=[True, False]`.
> - **NaN handling**: By default, `NaN` values are placed at the end (`na_position="last"`). You can switch this to `"first"` if required.
> - **Inplace mutation vs Chaining**: Avoid `inplace=True`. In modern Pandas (especially with Copy-on-Write), chaining without `inplace` is faster, safer, and prevents subtle reference bugs.

In [14]:
# Sort by release Year in descending order (newest first)
movies.sort_values(by="Year", ascending=False).head()

,Rank,Studio,Gross,Year
Title,,,,
Avengers: Endgame,1,Buena Vista,"$2,796.30",2019
John Wick: Chapter 3 - Parabellum,458,Lionsgate,$304.70,2019
The Wandering Earth,114,China Film Corporation,$699.80,2019
Toy Story 4,198,Buena Vista,$519.80,2019
How to Train Your Dragon: The Hidden World,199,Universal,$519.80,2019


In [15]:
# Sort by Studio (A-Z) and Year (descending: newest first)
movies.sort_values(by=["Studio", "Year"], ascending=[True, False]).head()

,Rank,Studio,Gross,Year
Title,,,,
The Blair Witch Project,588,Artisan,$248.60,1999
Avengers: Endgame,1,Buena Vista,"$2,796.30",2019
Captain Marvel,22,Buena Vista,"$1,128.30",2019
Aladdin,59,Buena Vista,$880.20,2019
Toy Story 4,198,Buena Vista,$519.80,2019


In [16]:
# Sort alphabetically by the row Index (Title)
movies.sort_index().head()

,Rank,Studio,Gross,Year
Title,,,,
"10,000 B.C.",536,Warner Brothers,$269.80,2008
101 Dalmatians,708,Buena Vista,$215.90,1961
101 Dalmatians,425,Buena Vista,$320.70,1996
2 Fast 2 Furious,632,Universal,$236.40,2003
2012,93,Sony,$769.70,2009


## 7. Categorical Frequency Analysis: `value_counts()`

> 💡 **Interview Deep-Dive — `value_counts()` Parameters**:
> - `normalize=True`: Returns relative frequencies (proportions / percentages) summing to 1.0. Extremely useful in exploratory data analysis and feature engineering.
> - `dropna=False`: By default, `value_counts()` **drops missing values (`NaN`)**! In interviews, always mention `dropna=False` to verify whether missing values dominate a categorical column.
> - `bins=N`: Discretizes continuous numeric columns into equal-width bins directly.

In [17]:
# Top 10 studios by total movie count
movies["Studio"].value_counts().head(10)

Studio
Warner Brothers    132
Buena Vista        125
Fox                117
Universal          109
Sony                86
Paramount           76
Dreamworks          27
Lionsgate           21
New Line            16
TriStar             11
Name: count, dtype: int64

In [18]:
# Pro-tip: Relative market share (% of catalog) per studio
movies["Studio"].value_counts(normalize=True).head(5) * 100

Studio
Warner Brothers    16.879795
Buena Vista        15.984655
Fox                14.961637
Universal          13.938619
Sony               10.997442
Name: proportion, dtype: float64

## 8. Vectorized Boolean Filtering & Logical Combinations

### ⚠️ Top Interview Trap: Bitwise Operators & Precedence
1. **Bitwise vs Boolean**: In Pandas, you **MUST** use bitwise operators (`&` for AND, `|` for OR, `~` for NOT).
   - Using Python's `and` / `or` fails with:
     ```text
     ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
     ```
   - *Reason*: Python's `and` evaluates the boolean truthiness of the entire container, whereas Pandas needs element-wise comparisons over vectors.
2. **Operator Precedence Trap**: In Python, bitwise operators (`&`, `|`) have **higher precedence** than comparison operators (`==`, `<`, `>`).
   - `movies["Studio"] == "Universal" & movies["Year"] == 2015` evaluates as:
     `movies["Studio"] == ("Universal" & movies["Year"]) == 2015`, causing a `TypeError`.
   - **Rule**: Every individual condition **MUST be wrapped in parentheses**: `(cond1) & (cond2)`.
3. **Range Filtering**: Use `.between(lower, upper, inclusive="both")` instead of `(col >= lower) & (col <= upper)` for cleaner and faster code.

In [19]:
# 1. Single condition filter
universal_movies = movies[movies["Studio"] == "Universal"]
universal_movies.head(3)

,Rank,Studio,Gross,Year
Title,,,,
Jurassic World,6,Universal,"$1,671.70",2015
Furious 7,8,Universal,"$1,516.00",2015
Jurassic World: Fallen Kingdom,13,Universal,"$1,309.50",2018


In [20]:
# 2. Logical AND (&) with mandatory parentheses
released_by_universal = movies["Studio"] == "Universal"
released_in_2015 = movies["Year"] == 2015

movies[released_by_universal & released_in_2015]

,Rank,Studio,Gross,Year
Title,,,,
Jurassic World,6,Universal,"$1,671.70",2015
Furious 7,8,Universal,"$1,516.00",2015
Minions,19,Universal,"$1,159.40",2015
Fifty Shades of Grey,165,Universal,$571.00,2015
Pitch Perfect 2,504,Universal,$287.50,2015
Ted 2,702,Universal,$216.70,2015
Everest,766,Universal,$203.40,2015
Straight Outta Compton,776,Universal,$201.60,2015


In [21]:
# 3. Logical OR (|)
movies[released_by_universal | released_in_2015].head()

,Rank,Studio,Gross,Year
Title,,,,
Star Wars: The Force Awakens,4,Buena Vista,"$2,068.20",2015
Jurassic World,6,Universal,"$1,671.70",2015
Furious 7,8,Universal,"$1,516.00",2015
Avengers: Age of Ultron,9,Buena Vista,"$1,405.40",2015
Jurassic World: Fallen Kingdom,13,Universal,"$1,309.50",2018


In [22]:
# 4. Numeric inequality (<)
before_1975 = movies["Year"] < 1975
movies[before_1975].head()

,Rank,Studio,Gross,Year
Title,,,,
The Exorcist,252,Warner Brothers,$441.30,1973
Gone with the Wind,288,MGM,$402.40,1939
Bambi,540,RKO,$267.40,1942
The Godfather,604,Paramount,$245.10,1972
101 Dalmatians,708,Buena Vista,$215.90,1961


In [23]:
# 5. Range filter using .between() [inclusive by default]
mid_80s = movies["Year"].between(1983, 1986)
movies[mid_80s].head()

,Rank,Studio,Gross,Year
Title,,,,
Return of the Jedi,222,Fox,$475.10,1983
Back to the Future,311,Universal,$381.10,1985
Top Gun,357,Paramount,$356.80,1986
Indiana Jones and the Temple of Doom,403,Paramount,$333.10,1984
Crocodile Dundee,413,Paramount,$328.20,1986


In [24]:
# 6. Multi-category matching using .isin() [preferred over chained | statements]
top_majors = movies["Studio"].isin(["Universal", "Warner Brothers", "Paramount"])
movies[top_majors].sample(4, random_state=42)

,Rank,Studio,Gross,Year
Title,,,,
Meet the Fockers,201,Universal,$516.60,2004
Dracula Untold,699,Universal,$217.10,2014
Batman v Superman: Dawn of Justice,64,Warner Brothers,$873.60,2016
2 Fast 2 Furious,632,Universal,$236.40,2003


## 9. Vectorized String Operations & Type Transformation

### 💡 Interview Pro-Tip: The `.str` Accessor & The Idempotency Pitfall
- In Pandas, string methods are vectorized via the `.str` accessor (e.g., `.str.lower()`, `.str.contains()`, `.str.replace()`).
- **NaN Handling**: Unlike pure Python `val.lower()`, Pandas `.str` methods safely propagate `NaN` values without throwing exceptions.
- **The Idempotency Pitfall**:
  - Overwriting `movies["Gross"] = movies["Gross"].str.replace(...).astype(float)` in a notebook cell mutates the column in-place.
  - If you re-run that cell, `movies["Gross"]` is already a `float64`, and calling `.str` raises:
    ```text
    AttributeError: Can only use .str accessor with string values!
    ```
  - **Modern Standard**: In Pandas 1.x, string columns were `object`. In Pandas 3.0+, they are native `str` / `StringDtype`. Using `pd.api.types.is_numeric_dtype(col)` provides an idempotent, version-agnostic check!

In [25]:
# Substring search on index: find all movies with "dark" in the title (case-insensitive)
has_dark_in_title = movies.index.str.lower().str.contains("dark", na=False)
movies[has_dark_in_title]

,Rank,Studio,Gross,Year
Title,,,,
Transformers: Dark of the Moon,23,Paramount,"$1,123.80",2011
The Dark Knight Rises,27,Warner Brothers,"$1,084.90",2012
The Dark Knight,39,Warner Brothers,"$1,004.90",2008
Thor: The Dark World,132,Buena Vista,$644.60,2013
Star Trek Into Darkness,232,Paramount,$467.40,2013
Fifty Shades Darker,309,Universal,$381.50,2017
Dark Shadows,600,Warner Brothers,$245.50,2012
Dark Phoenix,603,Fox,$245.10,2019


In [26]:
# Safe, idempotent conversion of 'Gross' from formatted currency string to float64
if not pd.api.types.is_numeric_dtype(movies["Gross"]):
    movies["Gross"] = (
        movies["Gross"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .astype(float)
    )

print("Gross Column Dtype:", movies["Gross"].dtype)
print("Average Gross Revenue ($M):", round(movies["Gross"].mean(), 2))

Gross Column Dtype: float64
Average Gross Revenue ($M): 439.03


## 10. Aggregations & GroupBy Basics: Split-Apply-Combine

### ⚠️ Top Interview Question: `groupby().count()` vs `groupby().size()`
- `.count()` calculates the number of **non-null (valid)** values in each column.
- `.size()` computes the total number of **rows** per group, **including nulls (`NaN`)**.
- In SQL terms:
  - `df.groupby("Studio").size()` $\Longleftrightarrow$ `SELECT Studio, COUNT(*) FROM movies GROUP BY Studio`
  - `df.groupby("Studio")["Gross"].count()` $\Longleftrightarrow$ `SELECT Studio, COUNT(Gross) FROM movies GROUP BY Studio`

In [27]:
# Group by Studio (creates a lazy DataFrameGroupBy object)
studios = movies.groupby("Studio")

In [28]:
# Count non-null gross values per studio (sorted descending)
studios["Gross"].count().sort_values(ascending=False).head(10)

Studio
Warner Brothers    132
Buena Vista        125
Fox                117
Universal          109
Sony                86
Paramount           76
Dreamworks          27
Lionsgate           21
New Line            16
TriStar             11
Name: Gross, dtype: int64

In [29]:
# Sum gross revenue per studio (sorted descending)
studios["Gross"].sum().sort_values(ascending=False).head(10)

Studio
Buena Vista        73585.0
Warner Brothers    58643.8
Fox                50420.8
Universal          44302.3
Sony               32822.5
Paramount          32486.0
Dreamworks         12260.4
Lionsgate          10033.2
New Line            6584.8
MGM                 3513.1
Name: Gross, dtype: float64

In [30]:
# Advanced: Multi-metric aggregation using .agg()
studios["Gross"].agg(
    movie_count="count",
    total_gross="sum",
    avg_gross="mean"
).sort_values(by="total_gross", ascending=False).head(10)

,movie_count,total_gross,avg_gross
Studio,,,
Buena Vista,125,73585.0,588.680000
Warner Brothers,132,58643.8,444.271212
Fox,117,50420.8,430.947009
Universal,109,44302.3,406.443119
Sony,86,32822.5,381.656977
Paramount,76,32486.0,427.447368
Dreamworks,27,12260.4,454.088889
Lionsgate,21,10033.2,477.771429
New Line,16,6584.8,411.550000


## 11. Quick Reference Cheat Sheet

| Task | Syntax | Key Notes / Gotchas |
| :--- | :--- | :--- |
| **Inspect Shape** | `df.shape` | Attribute, not method. Returns `(rows, cols)` |
| **Element Count** | `df.size` | Total cells ($N \times C$). `len(df)` is row count only. |
| **Positional Slice** | `df.iloc[start:stop]` | Half-open `[start, stop)`. Endpoint excluded! |
| **Label Slice** | `df.loc[start:stop]` | Closed `[start, stop]`. **Endpoint included!** |
| **Fast Scalar Lookup** | `df.at[label, col]` / `df.iat[i, j]` | 10x-50x faster than `.loc`/`.iloc` for single cells |
| **Frequency Count** | `s.value_counts(normalize=True)` | Use `dropna=False` to prevent hiding missing data |
| **Multi-Condition** | `(cond1) & (cond2)` | Must use `&`, `|`, `~` and parentheses around every clause |
| **Range Filter** | `s.between(a, b)` | Cleaner and faster than `(s >= a) & (s <= b)` |
| **Safe String Ops** | `s.str.lower()` | Handles `NaN` without crashing; requires object/string dtype |
| **Non-null vs Total** | `.count()` vs `.size()` | `.count()` skips `NaN`; `.size()` includes all rows |

---
## 🎯 12. Technical Interview Corner: Tricky Questions & Drills

### Q1: The Dreaded `SettingWithCopyWarning`
**Question**: What causes the `SettingWithCopyWarning` in Pandas? Why does chained indexing `df['A'][0] = 99` trigger it, and what is the idiomatic fix?

**Answer**:
- **Root Cause**: Chained indexing (`df[col][row]`) involves two separate operations:
  1. `df['A']`: Evaluated first. It may return either a **view** of the original memory or an independent **copy** (depending on memory layout and whether single or mixed dtypes exist).
  2. `[0] = 99`: Mutates the result of step 1. If step 1 returned a copy, your mutation is made to a temporary copy that gets immediately discarded, leaving the original `df` completely unmodified!
- **Idiomatic Solution**: Always use a single `.loc[row_indexer, col_indexer]` call to guarantee in-place assignment on the underlying DataFrame.

In [31]:
# Demonstration of SettingWithCopyWarning prevention
sample_df = movies.head(3).copy()

# ❌ INCORRECT (Chained assignment):
# sample_df["Gross"]["Titanic"] = 3000.0  # Raises SettingWithCopyWarning or fails silently

# ✅ CORRECT (Single .loc accessor):
sample_df.loc["Titanic", "Gross"] = 3000.0
sample_df.loc[["Titanic"], ["Gross"]]

,Gross
Title,
Titanic,3000.0


### Q2: `df['col']` vs `df[['col']]`
**Question**: What is the structural and dimensional difference between `df['Gross']` and `df[['Gross']]`?

**Answer**:
- `df['Gross']` (single bracket): Returns a **1D `pandas.Series`**. Shape is `(N,)`.
- `df[['Gross']]` (double bracket / list of columns): Returns a **2D `pandas.DataFrame`** containing a single column. Shape is `(N, 1)`.
- *Interview Tip*: Many machine learning pipelines (e.g. `scikit-learn`'s `transformer.fit_transform(X)`) require a 2D input matrix ($N \times 1$). Passing `df['Gross']` will raise a dimension error; passing `df[['Gross']]` works natively.

In [32]:
single_bracket = movies["Gross"]
double_bracket = movies[["Gross"]]

print(f"movies['Gross']   type: {type(single_bracket)} | shape: {single_bracket.shape}")
print(f"movies[['Gross']] type: {type(double_bracket)} | shape: {double_bracket.shape}")

movies['Gross']   type: <class 'pandas.Series'> | shape: (782,)
movies[['Gross']] type: <class 'pandas.DataFrame'> | shape: (782, 1)


### Q3: Slicing on an Integer Index
**Question**: Suppose a DataFrame `df` has an integer index `[1, 2, 3, 4, 5]`. What is the difference between `df.loc[1:3]` and `df.iloc[1:3]`?

**Answer**:
- `df.loc[1:3]`: Looks up index **labels** 1 through 3 **inclusive**. It returns rows with index labels `1, 2, 3` (3 rows).
- `df.iloc[1:3]`: Looks up 0-based **positions** 1 up to 3 **exclusive**. It returns the 2nd and 3rd rows (positions 1 and 2, matching labels `2, 3`).

In [33]:
# Drill: Integer Index ambiguity
demo_df = pd.DataFrame({"Value": [10, 20, 30, 40, 50]}, index=[1, 2, 3, 4, 5])
print("Original DataFrame with explicit integer index [1..5]:")
print(demo_df)

print("\n--- demo_df.loc[1:3] (Label based, inclusive): ---")
print(demo_df.loc[1:3])

print("\n--- demo_df.iloc[1:3] (Positional, exclusive): ---")
print(demo_df.iloc[1:3])

Original DataFrame with explicit integer index [1..5]:
   Value
1     10
2     20
3     30
4     40
5     50

--- demo_df.loc[1:3] (Label based, inclusive): ---
   Value
1     10
2     20
3     30

--- demo_df.iloc[1:3] (Positional, exclusive): ---
   Value
2     20
3     30


### Q4: Memory Optimization & The Categorical Dtype
**Question**: You are given a DataFrame with 20 million rows where a column `Studio` contains only 20 unique studio names stored as strings. How would you optimize memory usage and groupby speed?

**Answer**:
- Convert the column to the **`category`** dtype: `df['Studio'] = df['Studio'].astype('category')`.
- **Under the hood**: Pandas replaces repeated Python string objects with compact integer codes (`int8` or `int16`, 1-2 bytes per row) pointing to an internal dictionary of unique categories.
- **Benefits**:
  1. Often reduces column memory consumption by **80% to 95%**.
  2. Aggregations and grouping on integer codes are orders of magnitude faster than string hashing.

In [34]:
# Compare memory consumption of raw string vs category
raw_studio = movies["Studio"].copy()
cat_studio = raw_studio.astype("category")

raw_mem = raw_studio.memory_usage(deep=True)
cat_mem = cat_studio.memory_usage(deep=True)

print(f"Raw String Memory: {raw_mem:,} bytes")
print(f"Category Memory:   {cat_mem:,} bytes")
print(f"Memory Reduction:  {((raw_mem - cat_mem) / raw_mem) * 100:.1f}%")

Raw String Memory: 113,264 bytes
Category Memory:   71,036 bytes
Memory Reduction:  37.3%


### Q5: Hands-on Interview Coding Challenge
**Challenge**: In a single chained expression, find the top 3 studios with the highest **average gross revenue per movie**, considering only movies released **in 2000 or later**, and including only studios with **at least 5 movies** in that period.

In [35]:
# Solution to Coding Challenge using pure Pandas method chaining
challenge_result = (
    movies[movies["Year"] >= 2000]
    .groupby("Studio")["Gross"]
    .agg(movie_count="count", avg_gross="mean")
    .query("movie_count >= 5")
    .sort_values(by="avg_gross", ascending=False)
    .head(3)
    .round(2)
)

challenge_result

,movie_count,avg_gross
Studio,,
Buena Vista,99,646.40
New Line,10,490.02
Lionsgate,21,477.77
